In [0]:
import requests
import time
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta

API_URL = "https://api.retailops.com/v1/stores"
API_KEY = dbutils.secrets.get("retail_scope", "api_key")

# Determine incremental load timestamp
try:
    last_ts = spark.table("retailnew.bronze.store_inventory_api") \
                   .agg(F.max("api_updated_at")).collect()[0][0]
except:
    last_ts = None

updated_after = (last_ts or datetime.utcnow() - timedelta(days=7)).strftime("%Y-%m-%d")

def fetch_page(page):
    url = f"{API_URL}?updated_after={updated_after}&page={page}"
    headers = {"Authorization": f"Bearer {API_KEY}"}

    for attempt in range(3):
        try:
            r = requests.get(url, headers=headers, timeout=30)
            if r.status_code == 200:
                return r.json()
        except:
            pass
        time.sleep(2)

    raise Exception("API failed after 3 retries")

# Fetch all pages
first = fetch_page(1)
total_pages = first.get("total_pages", 1)
records = first["stores"]

for p in range(2, total_pages + 1):
    records.extend(fetch_page(p)["stores"])

# Convert to DataFrame
schema = StructType([
    StructField("store_id", StringType()),
    StructField("name", StringType()),
    StructField("address", StringType()),
    StructField("inventory", ArrayType(
        StructType([
            StructField("product_id", StringType()),
            StructField("qty", IntegerType()),
            StructField("updated_at", TimestampType())
        ])
    ))
])

df = spark.createDataFrame(records, schema)

df = (
    df.withColumn("item", F.explode("inventory"))
      .select(
          "store_id",
          "name",
          "address",
          F.col("item.product_id").alias("product_id"),
          F.col("item.qty").alias("qty"),
          F.col("item.updated_at").alias("api_updated_at"),
          F.current_timestamp().alias("ingest_ts")
      )
)

df.write.format("delta").mode("append").saveAsTable("retail.bronze.store_inventory_api")
